# 🚕 NYC Taxi — Kafka Ingestion Pipeline

This notebook downloads **one month** of NYC TLC Yellow Taxi trip data (Parquet),
converts it to JSON rows, and pushes them into a **Kafka topic** (`trips-raw`).

### Architecture
```
NYC TLC (Parquet on S3)
    │
    ▼
Download → Pandas DataFrame
    │
    ▼
Kafka Producer → Topic: trips-raw
    │
    ▼
(Next step: Spark Streaming Consumer → HDFS Bronze Layer)
```

### Prerequisites
- Master node running (`compose.master.yml`)
- At least one worker running (`compose.worker.yml`)
- Kafka broker accessible at `kafka:9092`

---
## 1. Configuration

In [1]:
# ── Configuration ────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP  = "kafka:9092"          # Kafka broker (internal docker network)
KAFKA_TOPIC      = "trips-raw"           # Topic to produce to

# NYC TLC data — Yellow Taxi, January 2024
YEAR  = 2024
MONTH = 1
DATA_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{YEAR}-{MONTH:02d}.parquet"

# Producer settings
BATCH_SIZE   = 500      # Send in batches for better throughput
LINGER_MS    = 50       # Wait up to 50ms to batch messages
MAX_ROWS     = None     # Set to e.g. 10000 for quick test, None = full month

print(f"📥 Data URL : {DATA_URL}")
print(f"📡 Kafka    : {KAFKA_BOOTSTRAP}")
print(f"📨 Topic    : {KAFKA_TOPIC}")

📥 Data URL : https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
📡 Kafka    : kafka:9092
📨 Topic    : trips-raw


---
## 2. Download the Parquet File

In [2]:
import pandas as pd
import time

print(f"⏳ Downloading {DATA_URL} ...")
start = time.time()

df = pd.read_parquet(DATA_URL)

elapsed = time.time() - start
print(f"✅ Downloaded in {elapsed:.1f}s")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head(3)

⏳ Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet ...


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

---
## 3. Explore the Data (Quick Look)

In [ ]:
print("── Column Types ──")
print(df.dtypes)
print()
print("── Null Counts ──")
print(df.isnull().sum())
print()
print("── Basic Stats ──")
df.describe()

---
## 4. Create Kafka Topic

In [ ]:
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP)

try:
    admin.create_topics([
        NewTopic(
            name=KAFKA_TOPIC,
            num_partitions=3,       # 3 partitions = one per Spark worker
            replication_factor=1    # Single broker, RF must be 1
        )
    ])
    print(f"✅ Topic '{KAFKA_TOPIC}' created (3 partitions)")
except TopicAlreadyExistsError:
    print(f"ℹ️  Topic '{KAFKA_TOPIC}' already exists")
finally:
    admin.close()

# List all topics to verify
from kafka import KafkaConsumer
consumer = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP)
print(f"📋 All topics: {sorted(consumer.topics())}")
consumer.close()

---
## 5. Produce Data to Kafka

In [ ]:
import json
from kafka import KafkaProducer

# ── Prepare the DataFrame ────────────────────────────────────────────────
# Convert datetime columns to ISO strings for JSON serialization
df_send = df.copy()
if MAX_ROWS:
    df_send = df_send.head(MAX_ROWS)
    print(f"⚠️  Limiting to {MAX_ROWS:,} rows for testing")

for col in df_send.select_dtypes(include=['datetime64']).columns:
    df_send[col] = df_send[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

total_rows = len(df_send)
print(f"📤 Producing {total_rows:,} rows to '{KAFKA_TOPIC}' ...")

In [ ]:
# ── Kafka Producer ───────────────────────────────────────────────────────
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    batch_size=16384 * 4,     # 64KB batches
    linger_ms=LINGER_MS,
    compression_type='gzip',  # Compress for network efficiency
    acks='all',               # Wait for broker acknowledgment
    retries=3,
    buffer_memory=67108864,   # 64MB buffer
)

errors = []
sent = 0
start = time.time()

def on_error(exc):
    errors.append(exc)

for idx, row in df_send.iterrows():
    record = {k: (None if pd.isna(v) else v) for k, v in row.items()}
    
    # Use VendorID as key for partitioning (distributes across 3 partitions)
    key = str(record.get('VendorID', ''))
    
    producer.send(KAFKA_TOPIC, key=key, value=record).add_errback(on_error)
    sent += 1
    
    if sent % 100_000 == 0:
        elapsed = time.time() - start
        rate = sent / elapsed
        pct = (sent / total_rows) * 100
        print(f"  ⏳ {sent:>10,} / {total_rows:,} ({pct:.1f}%)  |  {rate:,.0f} rows/sec  |  errors: {len(errors)}")

# Flush remaining messages
producer.flush()
producer.close()

elapsed = time.time() - start
print(f"\n{'='*60}")
print(f"✅ Done! Sent {sent:,} rows in {elapsed:.1f}s ({sent/elapsed:,.0f} rows/sec)")
print(f"❌ Errors: {len(errors)}")
if errors:
    print(f"   First error: {errors[0]}")

---
## 6. Verify — Read a Few Messages Back

In [ ]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset='earliest',
    consumer_timeout_ms=5000,    # Stop after 5s of no new messages
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    max_poll_records=5,
)

print(f"📖 Reading first few messages from '{KAFKA_TOPIC}' ...\n")
count = 0
for msg in consumer:
    print(f"  Partition={msg.partition}  Offset={msg.offset}")
    print(f"  Key={msg.key}")
    print(f"  Value={json.dumps(msg.value, indent=2)[:300]}")
    print()
    count += 1
    if count >= 5:
        break

consumer.close()
print(f"✅ Verified {count} messages successfully")

---
## 7. Topic Stats

In [ ]:
from kafka import KafkaConsumer, TopicPartition

consumer = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP)
partitions = consumer.partitions_for_topic(KAFKA_TOPIC)

total_messages = 0
print(f"📊 Topic: {KAFKA_TOPIC}")
print(f"{'─'*50}")

if partitions:
    tps = [TopicPartition(KAFKA_TOPIC, p) for p in sorted(partitions)]
    consumer.assign(tps)
    
    end_offsets = consumer.end_offsets(tps)
    beg_offsets = consumer.beginning_offsets(tps)
    
    for tp in tps:
        count = end_offsets[tp] - beg_offsets[tp]
        total_messages += count
        print(f"  Partition {tp.partition}: {count:>12,} messages  (offset {beg_offsets[tp]} → {end_offsets[tp]})")
    
    print(f"{'─'*50}")
    print(f"  Total:      {total_messages:>12,} messages")
else:
    print(f"  ⚠️ Topic '{KAFKA_TOPIC}' not found or has no partitions")

consumer.close()

---
## ✅ Summary

| Step | Status |
|------|--------|
| Download Parquet from NYC TLC | ✅ |
| Create Kafka topic `trips-raw` | ✅ |
| Produce all rows to Kafka | ✅ |
| Verify messages readable | ✅ |

### Next Steps
1. **`02_spark_streaming_bronze.ipynb`** — Spark Structured Streaming reads from `trips-raw` → writes Parquet to HDFS Bronze layer
2. **`03_spark_batch_silver.ipynb`** — ETL: clean, validate, deduplicate → HDFS Silver layer
3. **`04_spark_ml_gold.ipynb`** — Feature engineering + ML model training
4. **Airflow DAGs** — Orchestrate all steps automatically